# Task 1: File ingestion and regular expressions

## 1. Robust File Reader Function
The goal: 
Create a function that reads a file and
returns both its contents and the character encoding that worked.

In [73]:
from pathlib import Path

# Write the function
def read_file(file_path):
    """
    Reads a file using UTF-8 encoding, falling back to ISO-8859-1
    if UTF-8 decoding fails.

    Args:
        file_path: The path to the file to read.

    Returns:
        A tuple containing the file contents as a string and the
        encoding used ("UTF-8" or "ISO-8859-1").
    """

    file_path = Path(file_path)  # Convert the file path to a Path object

    try:
        text = file_path.read_text(encoding="utf-8") # reads the file with UTF-8
        return text, "UTF-8"                         # returns the text and encoding

    except UnicodeDecodeError:                            # if UTF-8 decoding fails, it falls back to ISO-8859-1
        text = file_path.read_text(encoding="ISO-8859-1") # reads the file with ISO-8859-1
        return text, "ISO-8859-1"                         # returns the text and encoding

    

## 2. Log Parser Function
The goal: 
Create a function that reads a log file, 
identifies valid sales transactions using regular expressions (re), 
returns the information structured as a list of dictionaries.

Valid line format:
[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)

Regular expression (Regex): 
The pattern used to find and extract the six fields from lines like these

In [74]:
import re 

# Write the function to parse the log file
def parse_log_file(file_text_content):
    """ 
    Parses the log file content and extracts transaction information.

    Args:
        file_text_content: The content of the log file as a string.

    Returns:
        A list of dictionaries, each containing information about a transaction.
    """

# Making the regex-pattern
    pattern = (    
        r"\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] "
        r"\[(POS_\d+)\] "
        r"item=(.*?) "
        r"price=(\d+\.\d{2}) "
        r"(EUR|NOK|USD) "
        r"\((SUCCESS|REFUND|CANCELLED)\)"
    )

    transactions = []

    for line in file_text_content.splitlines():
        match = re.search(pattern, line)        # searches for matches in the line using the regex pattern

        if match:
            timestamp, register_id, item, price, currency, status = match.groups() # extracts the matched groups from the regex match

        # Create dictionary (for transaction)
            transaction = {
                "timestamp": timestamp,
                "register_id": register_id,
                "item": item,
                "price": float(price), 
                "currency": currency,
                "status": status
            }

            transactions.append(transaction)  # adds the new dictionary to the transactions list.

    return transactions # When all lines have been examined, --> function returns the list containing all the transactions it found.

# Task 2: Recursive utility functions
## Part 1: directory search
in this code we find all .log files within the nested "data" directory


In [ ]:
from pathlib import Path

#function to find all .log files in a given folder and its subfolders, for the task this will be "data"
def find_log_files(folder_path):
    """
        Function recursively search a directory and its subdirectory for .log files
    

        Args:
        folder_path: The starting directory, provided as a string
            or Path object.

        Returns:
            A list of Path objects representing all .log files found
            in the directory and its subdirectories.


    """
    folder = Path(folder_path)
    log_files = []

    # recursive search for .log files
    for item in folder.iterdir():

        if item.is_dir():
            log_files.extend(find_log_files(item))  # recursive call for subdirectories

        elif item.is_file() and item.suffix == ".log":
            log_files.append(item)

    return log_files


#test the function by searching for .log files in the "data" folder
log_files = find_log_files("data")

print(f"Found {len(log_files)} log files:")

for log_file in log_files:
    print(log_file)


Found 9 log files:
data\2026\library_cafe\POS_01\register_sales.log
data\2026\library_cafe\POS_02\register_sales.log
data\2026\library_cafe\POS_03\register_sales.log
data\2026\science_building\POS_01\register_sales.log
data\2026\science_building\POS_02\register_sales.log
data\2026\science_building\POS_03\register_sales.log
data\2026\student_union\POS_01\register_sales.log
data\2026\student_union\POS_02\register_sales.log
data\2026\student_union\POS_03\register_sales.log


## Part 2: Recursive search through JSON data

identifying active register IDs from the nested json file

In [ ]:
import json

with open("data/store_config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

def find_active_registers(data) -> set[str]:
    """
        Recursively identify active register IDs in nested JSON data

        Args:
            data: Parsed JSON data containing nested dictionaries,
                lists and other values.

        Returns:
            A set of unique register ID strings where the "active"
            property is True.
    
    """

    active_registers = set()

    #check if the dictionairy represents an active register and search through its values recursively
    if isinstance(data, dict):
        if data.get("active") is True and "id" in data:
            active_registers.add(data["id"])

        for value in data.values():
            active_registers.update(find_active_registers(value))

    #search through each element recursively
    elif isinstance(data, list):
        for item in data:
            active_registers.update(find_active_registers(item))

    return active_registers 


#test the function by finding all active registers in the store_config.json file
active_registers = find_active_registers(config)
print("Active registers:", active_registers)
print("Number of active registers:", len(active_registers))



Active registers: {'POS_02', 'POS_03', 'POS_01'}
Number of active registers: 3


# Task 3: Functions for Cleaning and Analysing Data using NumPy

## 1. Function for Removing Price Outliers 
The function should perform four tasks:
1. Converts the input prices into a 1D NumPy array of floats.
2. Finds outliers that are more than 3 standard deviations from the mean.
3. Replaces outliers with the median of the valid prices.
4. Returns the cleaned array. 


In [77]:
import numpy as np

# Write the function to clean price outliers
def clean_price_outliers(prices):
    """Cleans price outliers from a list of prices.
    
    Args:
        prices: A list or array of prices (floats or ints).

    Returns:
        A numpy array of prices with outliers replaced by the median price.
    """
    prices = np.asarray(prices, dtype=float, copy=True)  # Convert the input into a NumPy array, dtype=float ensures that the values are floats

# Calculate the mean, standard deviation, and median of the prices
    mean = np.mean(prices)
    std = np.std(prices)
    median = np.median(prices)

# Find outliers: prices that are more than 3 standard deviations away from the mean
    outliers = np.abs(prices - mean) > 3 * std

# Replaces outliers with the median
    prices[outliers] = median

# Returns the cleaned 1D array
    return prices

## 2. Function: Calculates the Proportion of Sales that are Refunds

Function: compute_refund_ratio(sales_count, refund_count)

--> Divide the number of refunds by the number of sales, element by element, and add \(10^{-6}\) to the denominator to avoid division by zero.

Formula:  Refund Ratio = Refund Count/Sales Count + 10^-6


In [78]:
# Write function to compute refund ratio
def compute_refund_ratio(
    sales_count: np.ndarray,
    refund_count: np.ndarray
) -> np.ndarray:
    """
    Computes the refund ratio for each register.

    Args:
        sales_count: A numpy array containing the number of sales for each register.
        refund_count: A numpy array containing the number of refunds for each register. 

    Returns:
        A numpy array containing the refund ratio for each register.
    """
    
    epsilon = 10**-6   # Saves the value of 10^-6 in a variable called epsilon. 

    refund_ratio = refund_count / (sales_count + epsilon) # formula for calculating the refund ratio, adding epsilon to avoid division by zero

    return refund_ratio  # the result is returned as a numpy array containing the refund ratio for each register.

In [79]:
from pathlib import Path

print("Working directory:", Path.cwd())
print("Data location:", Path("data").resolve())

log_files = find_log_files("data")
print("Log files found:", len(log_files))

transactions = []

for file in log_files:
    file_content, encoding = read_file(file)
    parsed = parse_log_file(file_content)
    transactions.extend(parsed)

print("Transactions parsed:", len(transactions))

Working directory: c:\Users\hansk\OneDrive - Norwegian University of Life Sciences\STUDIER - ØLIT\INF201 - Videregående Programmering\deliverable_2
Data location: C:\Users\hansk\OneDrive - Norwegian University of Life Sciences\STUDIER - ØLIT\INF201 - Videregående Programmering\deliverable_2\data
Log files found: 9
Transactions parsed: 720


# Task 4: Data analysis using Pandas 
Transform transaction records into a structured Pandas DataFrame and perform an analyses 

### 4.1 create a DataFrame to sort the transactions from the log files


In [ ]:
import pandas as pd
import numpy as np

transactions = []

# function to create and organize the Dataframe
def create_dataframe(transactions: list[dict]) -> pd.DataFrame:
    """
    Convert a list of transaction dictionaries into a Pandas DataFrame.

    Args:
        transactions: A list of dictionaries containing parsed
        transaction details, including timestamps, items,
        prices, currencies and statuses.

    Returns:
        A DataFrame with timestamps converted to datetime,
        set as the index, and sorted chronologically

    """
    #convert the list of dictionaries into a pandas DataFrame
    df = pd.DataFrame(transactions)

    #convert the timestamp column to datetime format
    df['timestamp'] = pd.to_datetime(df['timestamp'])

    #set the timestamp column as the index of the DataFrame
    df.set_index('timestamp', inplace=True)

    #sort the DataFrame chronologically by timestamp
    df = df.sort_index()

    return df

### 4.2 group data by items and resample

In [81]:
def hourly_revenue(df: pd.DataFrame) -> pd.DataFrame:
    """Calculate total sales for each item in 1-hour intervals."""

    #group the DataFrame by the "item" column
    grouped_df = df.groupby('item')

    #resample transactions in 1-hour intervals and calculate the total sales for each item
    grouped_df = grouped_df.resample('1h').agg({'price': 'sum'})

    return grouped_df


### 4.3 create a pivot table displaying total transaction revenue


In [82]:
#create a pivot table to calculate total sales for each item across status categories
def status_revenue_pivot(df: pd.DataFrame) -> pd.DataFrame:
    """Calculate total sales for each item category."""

    #creates the pivot table
    pivot_df = pd.pivot_table(
        df,
        values='price',
        index='item',
        columns='status',
        aggfunc='sum',
        fill_value=0
    )

    return pivot_df

### Taks 4.4: Calculating refund ratios for each item

create a function that calculates the refund ratio, we also include the refund count and sales count for better interpretation 


In [83]:
def refund_ratio_per_item(df: pd.DataFrame) -> pd.DataFrame:
    """Calculate the refund ratio for each item."""

    #group the DataFrame by the "item" column
    grouped_df = df.groupby('item')

    #calculate the number of sales and refunds for each item
    sales_count = grouped_df['status'].apply(lambda x: (x == 'SUCCESS').sum())
    refund_count = grouped_df['status'].apply(lambda x: (x == 'REFUND').sum())

    #compute the refund ratio using the compute_refund_ratio function
    refund_ratio = compute_refund_ratio(sales_count.to_numpy(), refund_count.to_numpy())

    #create a new DataFrame to store the results
    result_df = pd.DataFrame({
        'sales_count': sales_count,
        'refund_count': refund_count,
        'refund_ratio': refund_ratio
    })

    return result_df


# Task 5: Creating the full workflow


In [ ]:
from pathlib import Path
import json

def run_workflow(data_folder: str | Path = "data") -> pd.DataFrame:
    """
    Runs the entire workflow: reads log files, parses them, creates a DataFrame,
    calculates hourly revenue, status revenue pivot, and refund ratios.
    the function combines the methods developed in tasks 1-4.

    Args:
        data_folder: The folder containing the log files.

    Returns:   
        A DataFrame containing cleaned and corrected transaction written to csv files

    Outputs:
        cleaned_transactions.csv
        hourly_revenue.json
        status_pivot_table.csv
        item_refund_ratios.csv

    """

    data_folder = Path(data_folder)

    #find log files and active registers 
    log_files = find_log_files(data_folder)

    with open(data_folder / "store_config.json", "r", encoding="utf-8") as f:
        config = json.load(f)

    active_registers = find_active_registers(config)


    transactions = []

    #read and parse transactions
    for file in log_files:
        file_content, encoding = read_file(file)
        transactions.extend(parse_log_file(file_content))

    df = create_dataframe(transactions)

    #replace original prices with cleaned values
    df["price"] = clean_price_outliers(df["price"].to_numpy())

    #Correct NOK prices
    df.loc[df["currency"] == "NOK", "price"] *= 10.0

    #Generate reports
    hourly_df = hourly_revenue(df)
    pivot_df = status_revenue_pivot(df)
    refund_df = refund_ratio_per_item(df)

    output_dir = data_folder.parent

    #Export output files
    df.reset_index().to_csv("cleaned_transactions.csv", index=False)

    hourly_df.reset_index().to_json(
        "hourly_revenue.json",
        orient="records",
        date_format="iso",
        indent=2
    )

    pivot_df.to_csv("status_pivot_table.csv")

    refund_df.to_csv("item_refund_ratios.csv")

    return df


df = run_workflow("data")
display(df.head())
    

,register_id,item,price,currency,status
timestamp,,,,,
2026-10-15 07:00:53,POS_01,Muffin,39.90,NOK,REFUND
2026-10-15 07:01:01,POS_02,Cappuccino,39.80,NOK,SUCCESS
2026-10-15 07:01:07,POS_02,Sandwich,78.10,NOK,SUCCESS
2026-10-15 07:01:09,POS_03,Croissant,41.00,NOK,SUCCESS
2026-10-15 07:01:12,POS_02,Cappuccino,4.07,EUR,SUCCESS
